# 06 Extract label time courses from evokeds

This notebook extracts label-wise time courses from saved source estimates (STCs).

Inputs per recording/condition:

- evoked source estimates from `04_apply_inverse_evokeds.ipynb`,
- subject-specific labels/annotations from FreeSurfer.

Outputs are written to `derivatives/meeg-pipeline/sub-*/meg/label_time_course/`.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    extract_label_time_courses_for_recordings,
    label_time_course_config_to_dataframe,
    label_time_course_input_overview_to_dataframe,
    label_time_course_qc_to_dataframe,
    label_time_course_results_to_dataframe,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)


def find_project_root(start: Path | None = None) -> Path:
    """Find the project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)


## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Selection

`iter_recordings(..., "all")` uses existing raw-BIDS recordings and excludes `sub-emptyroom` by default.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_LABEL_TIME_COURSE_QC = True
MAX_LABEL_TIME_COURSE_QC_ROWS = None  # None = all, or e.g. 4 for a quick test

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)


## Overwrite policy

Default: skip existing label time courses.

Set `OVERWRITE_STEPS = ["label_time_course"]` to recompute existing outputs.

In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "label_time_course",
            "overwrite": should_overwrite("label_time_course", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "label_time_course",
                OVERWRITE_STEPS,
            ),
        }
    ]
)


## Label-time-course parameters

Methodological parameters are read from `configs/local.yaml` under `source`.

In [ ]:
label_time_course_config_to_dataframe(config)

## Effective parameters

In [ ]:
METHOD = config.source.apply_inverse.method
PICK_CONDITIONS = config.source.apply_inverse.pick_conditions
PARCELLATION = config.source.labels.parcellation
EXTRACT_MODE = config.source.labels.extract_mode
TARGET_LABELS = config.source.labels.target_labels
SOURCE_SPACING = config.source.spacing

pd.DataFrame(
    [
        {
            "method": METHOD,
            "pick_conditions": PICK_CONDITIONS,
            "parcellation": PARCELLATION,
            "extract_mode": EXTRACT_MODE,
            "target_labels": TARGET_LABELS,
            "source_spacing": SOURCE_SPACING,
        }
    ]
)


## Input overview

This table checks whether source estimates and labels exist before extracting label time courses.

In [ ]:
label_time_course_policy = existing_output_policy_for_step(
    "label_time_course",
    OVERWRITE_STEPS,
)

label_time_course_overview = label_time_course_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=label_time_course_policy,
    method=METHOD,
    pick_conditions=PICK_CONDITIONS,
    parcellation=PARCELLATION,
    extract_mode=EXTRACT_MODE,
    target_labels=TARGET_LABELS,
)

label_time_course_overview


## Status summary

In [ ]:
if label_time_course_overview.empty:
    pd.DataFrame()
else:
    (
        label_time_course_overview
        .groupby(["status"], dropna=False)
        .size()
        .reset_index(name="n_jobs")
        .sort_values(["status"])
    )


## Ready jobs

In [ ]:
ready_label_time_course_jobs = label_time_course_overview.query("status == 'ready'").copy()

columns = [
    "subject",
    "session",
    "task",
    "run",
    "condition",
    "stc_path",
    "parcellation",
    "extract_mode",
    "n_labels",
    "ltc_path",
]
existing_columns = [column for column in columns if column in ready_label_time_course_jobs.columns]
ready_label_time_course_jobs[existing_columns]


## Extract label time courses

This cell processes all selected recordings and conditions. Existing outputs are skipped unless `OVERWRITE_STEPS` contains `"label_time_course"`. Missing inputs and per-job failures are returned as status rows rather than stopping the whole batch.

In [ ]:
label_time_course_results = extract_label_time_courses_for_recordings(
    config,
    selected_recordings,
    on_existing=label_time_course_policy,
    method=METHOD,
    pick_conditions=PICK_CONDITIONS,
    parcellation=PARCELLATION,
    extract_mode=EXTRACT_MODE,
    target_labels=TARGET_LABELS,
    spacing=SOURCE_SPACING,
    allow_empty=False,
    verbose=True,
)

label_time_course_results_df = label_time_course_results_to_dataframe(label_time_course_results)
label_time_course_results_df


## Result summary

In [ ]:
if "label_time_course_results_df" not in globals() or label_time_course_results_df.empty:
    pd.DataFrame()
else:
    (
        label_time_course_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_jobs")
        .sort_values(["status"])
    )


## Batch QC

Reads saved label-time-course tables and returns a compact QC table. This is non-interactive and safe for batch runs.

In [ ]:
if RUN_LABEL_TIME_COURSE_QC:
    if "label_time_course_results_df" not in globals() or label_time_course_results_df.empty:
        label_time_course_qc_status = pd.DataFrame(
            [{"status": "no_results", "message": "Run the extraction cell first."}]
        )
    else:
        candidate_results = label_time_course_results_df[
            label_time_course_results_df["status"].isin(
                ["written", "skipped_existing", "exists"]
            )
        ].copy()

        if candidate_results.empty:
            candidate_results = label_time_course_results_df.copy()

        label_time_course_qc_status = label_time_course_qc_to_dataframe(
            candidate_results,
            max_rows=MAX_LABEL_TIME_COURSE_QC_ROWS,
        )

    label_time_course_qc_status
else:
    print("Skipped label-time-course QC.")


## Expected outputs

Label time courses are written to:

```text
derivatives/meeg-pipeline/sub-*/meg/label_time_course/*_space-label_parc-*_desc-*-ltc.tsv
```

Each main table has labels as rows and time points as columns. Two sidecars are written next to it:

```text
*-labels.tsv
*-times.tsv
```